# QKSR Failure Localization — Kaggle Offline

Internet **OFF**, GPU **ON**. Attach the source ZIP, CIFAR-100 (or prepared ImageNet-A/R train/test folders), the existing pretrained ViT-B/16 weights, and any missing dependency wheels as Kaggle input datasets. No downloads, hyperparameter tuning or simulated benchmark results. All new outputs go to `/kaggle/working/`.

Read `docs/KAGGLE_OFFLINE_FAILURE_LOCALIZATION.md` in the source bundle for dataset formats, metric definitions and the files to return. Notebook cells contain no saved results.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import uuid
import zipfile

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
SOURCE_PATH = None  # Optional full path to the ZIP or its extracted qg_rsa directory.
os.environ.update({"HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "HF_DATASETS_OFFLINE": "1"})
marker = "qg_rsa/scripts/run_failure_localization.py"
if SOURCE_PATH:
    candidates = [Path(SOURCE_PATH)]
else:
    candidates = []
    for path in INPUT_ROOT.rglob("*.zip"):
        with zipfile.ZipFile(path) as archive:
            if marker in archive.namelist():
                candidates.append(path)
    candidates += [p.parent for p in INPUT_ROOT.rglob("BUILD_MANIFEST.json")
                   if (p.parent / "scripts/run_failure_localization.py").is_file()]
if len(candidates) != 1:
    raise RuntimeError(f"Expected one source bundle; set SOURCE_PATH explicitly. Found: {candidates}")
code_dir = WORKING_ROOT / ("qg_rsa_code_" + uuid.uuid4().hex[:8])
code_dir.mkdir(parents=True, exist_ok=False)
if candidates[0].is_dir():
    shutil.copytree(candidates[0], code_dir / "qg_rsa",
                    ignore=shutil.ignore_patterns("__pycache__", ".git", "logs", "ckpt"))
else:
    with zipfile.ZipFile(candidates[0]) as archive:
        for member in archive.infolist():
            target = (code_dir / member.filename).resolve()
            if not target.is_relative_to(code_dir.resolve()):
                raise RuntimeError("Unsafe source ZIP member")
        archive.extractall(code_dir)
REPO_ROOT = code_dir / "qg_rsa"
build = json.loads((REPO_ROOT / "BUILD_MANIFEST.json").read_text())
for relative, expected in build["source_sha256"].items():
    if hashlib.sha256((REPO_ROOT / relative).read_bytes()).hexdigest() != expected:
        raise RuntimeError(f"Source bundle checksum mismatch: {relative}")
sys.path.insert(0, str(REPO_ROOT))
print("Source verified:", REPO_ROOT)

In [ ]:
# Keep Kaggle's CUDA torch/torchvision. Install missing small dependencies only
# from uploaded wheels. --no-index prevents package downloads.
import importlib
for module in ("torch", "torchvision"):
    importlib.import_module(module)
missing = []
for module in ("timm", "easydict", "scipy", "tqdm"):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(module)
if missing:
    wheel_dirs = sorted({p.parent for p in INPUT_ROOT.rglob("*.whl")})
    if not wheel_dirs:
        raise RuntimeError(f"Upload offline dependency wheels for: {missing}")
    command = [sys.executable, "-m", "pip", "install", "--no-index",
               "--disable-pip-version-check", "--no-cache-dir"]
    for directory in wheel_dirs:
        command += ["--find-links", str(directory)]
    command += ["-r", str(REPO_ROOT / "requirements-kaggle-offline.txt")]
    subprocess.run(command, check=True)
import torch
print("CUDA:", torch.cuda.is_available(), "GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Configure assets and perform offline preflight

Leave paths as `None` for discovery. Ambiguous uploads require an explicit path. CIFAR uses the official `cifar-100-python` layout; its uploaded official archive can be extracted into a new writable directory. ImageNet-A/R requires an existing `train/` and `test/` split with identical class folders; this notebook never creates a new split.

In [ ]:
from argparse import Namespace
from scripts.run_failure_localization import prepare, run_study
from utils.offline_assets import resolve_data_root

DATASET = "cifar224"  # Or "imageneta" / "imagenetr", using that dataset's current config.
DATA_ROOT = None     # CIFAR: parent of cifar-100-python; ImageNet: parent of train/ and test/.
PRETRAINED_PATH = None  # Explicit .pth/.pt/.bin/.safetensors/.npz path if necessary.
CIFAR_ARCHIVE = None    # Optional full path to the official CIFAR-100 Python tar.gz.
if DATASET == "cifar224" and DATA_ROOT is None and CIFAR_ARCHIVE is None:
    try:
        resolve_data_root(DATASET, INPUT_ROOT)
    except ValueError as error:
        archives = list(INPUT_ROOT.rglob("cifar-100-python.tar.gz"))
        if len(archives) != 1 or "found 0" not in str(error):
            raise
        CIFAR_ARCHIVE = str(archives[0])
STUDY_DIR = prepare(Namespace(
    source=None, dataset=DATASET, input_root=str(INPUT_ROOT),
    output_root=str(WORKING_ROOT), data_root=DATA_ROOT,
    pretrained_path=PRETRAINED_PATH, cifar_archive=CIFAR_ARCHIVE,
))
# This has validated source/config/data/weights; it has NOT trained a benchmark.

## Run complete experiments

The default selection runs all nine experiments sequentially on one GPU. For session limits, select one seed or one variant and run complete experiments in separate notebook sessions. Keep the same source bundle, assets and runtime across sessions. There are no intermediate model checkpoints or task resumes. Repeating an already-started run is rejected; prepare a new study for a fresh rerun.

In [ ]:
SELECT_VARIANTS = ["A0", "A1", "A2"]
SELECT_SEEDS = [1993, 2015, 2026]  # Example per session: [1993]
run_study(STUDY_DIR, SELECT_VARIANTS, SELECT_SEEDS)
# One accuracy line per completed task plus a final accuracy summary. Final checkpoint only.

## Collect files to send back

Return the ZIP produced below (one ZIP per session if splitting the study). It contains configs, asset/runtime provenance, task CSV/JSON, per-run summary and logs. Large model checkpoints are excluded; download them separately if you want to retain trained models. Incomplete collections are explicitly marked `partial`. This notebook does not generate the comparison report.

In [ ]:
from scripts.collect_failure_localization import collect
RESULTS_ZIP = WORKING_ROOT / (STUDY_DIR.name + "_results_" + uuid.uuid4().hex[:8] + ".zip")
collect(STUDY_DIR, RESULTS_ZIP)
print("Send this results ZIP:", RESULTS_ZIP)
print("Final model files:", sorted(str(p) for p in (STUDY_DIR / "runs").glob("*/final_task.pkl")))